# Maps

Annual-mean maps from the HAMOCC annual history files (`hbgcy`): surface fields and depth slices, shown as model | observations | model - observations. Observations are interpolated onto the model grid. If the recipe has an `experiment_cntl` control case, it takes the place of the observations: test | control | test - control.

In [ ]:
import matplotlib.pyplot as plt
import yaml
from pyprojroot import here

from blomster import grid
from hamocc import climatology, compare, grids, io, obs, plots, timeseries, variables

In [ ]:
# Depth slices to show (m); must match levels of the model's z-level fields.
depths = list(variables.DEPTH_LEVELS_M)
compare_obs = True
compare_cntl = True  # use the control case (recipe `experiment_cntl`) when there is one

In [ ]:
with open(here('notebooks_hamocc/settings.yml')) as f:
    cfg = yaml.safe_load(f)
exp = cfg['experiment']
dsets = cfg['datasets']

lat, lon, _, clat, clon, _ = grid.load(grids.grid_file(cfg))

names = [n for n in list(variables.SURFACE_VARS) + ['co2fxn'] + list(variables.LEVEL_VARS)]
needed = sorted({h for n in names for h in climatology.history_names(n)} | {'depth_bnds'})
variables_needed = [n for n in needed if n != 'depth_bnds']

def open_case(case):
    files = io.history_files(case['input_root'], case['caseid'], 'hbgcy', *io.period(case, 'climo'))
    return io.open_available(files, variables_needed)[0]

ds = open_case(exp)
cntl = compare.control_case(cfg)
kind = compare.reference_kind(cfg, compare_obs, compare_cntl)
ds_cntl = open_case(cntl) if kind == 'control' else None

# plot only what the file(s) hold; say what is skipped
names, skipped = io.usable(ds, names, climatology.history_names)
if ds_cntl is not None:
    names, skipped_cntl = io.usable(ds_cntl, names, climatology.history_names)
    skipped.update(skipped_cntl)
io.report_skipped(skipped)

parea = timeseries.load_parea(grids.grid_file(cfg))

def obs_on_model(name, depth=None):
    spec = dsets['maps'].get(name)
    if spec is None:
        return None
    return obs.regrid_to_model(obs.load_obs(spec, dsets['obs_root'], depth), lat, lon)

def reference(name, depth=None):
    """Field the test case is compared with (control or obs, per `kind`), or None."""
    if kind == 'control':
        return climatology.annual_mean_map(ds_cntl, name, depth).load()
    return obs_on_model(name, depth) if kind == 'obs' else None

def show(name, depth=None):
    model = climatology.annual_mean_map(ds, name, depth).load()
    title = model.attrs['long_name'] + (f', {depth} m' if depth is not None else '')
    ref = reference(name, depth)
    if kind == 'control':
        fig = plots.map_panels(clon, clat, model, control=ref, title=title, area=parea)
    else:
        fig = plots.map_panels(clon, clat, model, ref, title=title, area=parea)
    plt.show()

```{note}
Observation units and scale factors in `recipe_hamocc.yml` are not yet verified against the files (`datasets.units_verified: false`). Check the difference panels before interpreting them quantitatively.
```

## Surface fields

In [ ]:
for name in [n for n in list(variables.SURFACE_VARS) + ['co2fxn'] if n in names]:
    show(name)

## Depth slices

In [ ]:
for name in [n for n in variables.LEVEL_VARS if n in names]:
    for depth in depths:
        show(name, depth)